In [1]:
import pandas as pd
import numpy as np
from workload_data_loader import load_workload_instance

In [2]:
n, m, utilities, demands, capacities = load_workload_instance("multi_resource_n28_m2.txt")


df = pd.DataFrame({"utility": utilities, **{f"resource_{i+1}": demands[i] for i in range(m)}})
print(demands)

[[ 45.   0.  85. 150.  65.  95.  30.   0. 170.   0.  40.  25.  20.   0.
    0.  25.   0.   0.  25.   0. 165.   0.  85.   0.   0.   0.   0. 100.]
 [ 30.  20. 125.   5.  80.  25.  35.  73.  12.  15.  15.  40.   5.  10.
   10.  12.  10.   9.   0.  20.  60.  40.  50.  36.  49.  40.  19. 150.]]


In [3]:
def evaluate(solution):
    # m is the number of resource bounds, in this case it will be 2
    # sum(iterable, start), if no np.zeros, the weight will be out of bounds as a set
    weight = sum((demands[:, i] for i in solution), np.zeros(m))
    value = sum(utilities[i] for i in solution)
    return weight, value

def fits(weight):
    return np.all(weight <= capacities)  # checks if all the resources under the capacity

solution = set()  #Set up a solution set, empty

while True:
    current_weight, current_value = evaluate(solution) #evaluate the current weight and value (resource and value)
    best_neighbor = solution #convert to best neighbor and best value to compare
    best_value = current_value

    # Neighbors: add, remove, or swap one model
    for i in range(n):  # STEP 3: generate neighbors   #n: the number of models, in this case: 28
        if i not in solution:
            candidate = solution | {i}  # Add the neighbor
            weight, value = evaluate(candidate)

            #if the candidate fits and value over best value, change the solution and end the for loop
            if fits(weight) and value > best_value:
                best_neighbor = candidate
                best_value = value

        else:
            candidate = solution - {i}  # Remove
            weight, value = evaluate(candidate)

            if fits(weight) and value > best_value: #update the best_neighbor and best value again to compare with the solution again
                best_neighbor = candidate
                best_value = value


            for j in range(n):
                if j not in solution:
                    candidate = (solution - {i}) | {j}  # Swap the resources values
                    weight, value = evaluate(candidate)

                    if fits(weight) and value > best_value:
                        best_neighbor = candidate
                        best_value = value

    # Stop if no improvement
    if best_neighbor == solution:
        break

    solution = best_neighbor

# Show result
weight, value = evaluate(solution)

print("Selected models:")
for i in sorted(solution):
    print("Model", i, "- Demands:", demands[:, i], "Utility:", utilities[i])

print("Total resource used:", weight, "/ Capacity:", capacities)
print("Total utility:", value)


Selected models:
Model 2 - Demands: [ 85. 125.] Utility: 22507.0
Model 4 - Demands: [65. 80.] Utility: 14148.0
Model 6 - Demands: [30. 35.] Utility: 4650.0
Model 7 - Demands: [ 0. 73.] Utility: 30800.0
Model 9 - Demands: [ 0. 15.] Utility: 4975.0
Model 10 - Demands: [40. 15.] Utility: 1160.0
Model 13 - Demands: [ 0. 10.] Utility: 11880.0
Model 18 - Demands: [25.  0.] Utility: 110.0
Model 20 - Demands: [165.  60.] Utility: 24355.0
Model 22 - Demands: [85. 50.] Utility: 11748.0
Model 23 - Demands: [ 0. 36.] Utility: 4550.0
Total resource used: [495. 499.] / Capacity: [500. 500.]
Total utility: 130883.0
